# P01：数据清洗与存储

本Notebook完成以下任务：
1. 单表清洗（缺失值、日期格式、数据类型、重复值、离群值）
2. 宽表与长表转换
3. 多表合并
4. 数据存储（CSV + Parquet进阶格式）

## 1. 环境准备

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from datetime import datetime

# 设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# 定义路径
data_dir = "data"
stock_dir = os.path.join(data_dir, "stock")
index_dir = os.path.join(data_dir, "index")
macro_dir = os.path.join(data_dir, "macro")
finance_dir = os.path.join(data_dir, "finance")
clean_dir = os.path.join(data_dir, "clean")
combined_dir = os.path.join(data_dir, "combined")

# 股票列表
stock_list = [
    {"code": "600036", "name": "招商银行", "industry": "银行"},
    {"code": "601398", "name": "工商银行", "industry": "银行"},
    {"code": "002594", "name": "比亚迪", "industry": "汽车"},
    {"code": "600104", "name": "上汽集团", "industry": "汽车"},
    {"code": "000002", "name": "万科A", "industry": "房地产"},
    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "000858", "name": "五粮液", "industry": "白酒"},
    {"code": "601857", "name": "中国石油", "industry": "能源"},
    {"code": "000063", "name": "中兴通讯", "industry": "通讯"},
    {"code": "002352", "name": "顺丰控股", "industry": "物流"},
]

stock_info_df = pd.DataFrame(stock_list)
print("环境准备完成！")

## 2. 单表清洗

### 2.1 缺失值检测

In [ ]:
# 读取所有股票数据并检测缺失值
missing_stats = []
stock_data = {}

for stock in stock_list:
    code = stock["code"]
    file_path = os.path.join(stock_dir, f"stock_{code}.csv")
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        stock_data[code] = df
        
        # 计算缺失值统计
        for col in df.columns:
            missing_count = df[col].isna().sum()
            missing_pct = missing_count / len(df) * 100
            missing_stats.append({
                'code': code,
                'name': stock['name'],
                'column': col,
                'missing_count': missing_count,
                'missing_pct': round(missing_pct, 2)
            })

# 创建缺失值统计表
missing_df = pd.DataFrame(missing_stats)
print("缺失值统计表：")
display(missing_df.pivot_table(index=['code', 'name'], columns='column', values='missing_count'))

In [ ]:
# 缺失值原因分析
print("\n缺失值可能原因分析：")
print("-" * 60)
total_missing = missing_df['missing_count'].sum()
print(f"总缺失值数量: {total_missing}")

if total_missing > 0:
    print("""
缺失值可能原因：
1. 停牌：股票因重大事项停牌期间没有交易数据
2. 节假日：A股市场在法定节假日不开盘
3. 数据源问题：akshare接口可能存在数据缺失
4. 新股上市：部分股票可能在2020年后才上市

处理策略：
- 对于价格数据的缺失，采用向前填充(ffill)方法
- 这样可以保持时间序列的连续性，同时避免引入未来信息
""")
else:
    print("所有股票数据完整，无缺失值。")

### 2.2 缺失值处理

In [ ]:
# 对每只股票进行缺失值处理
cleaned_stock_data = {}

print("缺失值处理：")
print("-" * 60)

for code, df in stock_data.items():
    original_len = len(df)
    missing_before = df.isna().sum().sum()
    
    # 向前填充缺失值
    df_cleaned = df.fillna(method='ffill')
    
    # 如果仍有缺失值（开头部分），用向后填充
    df_cleaned = df_cleaned.fillna(method='bfill')
    
    missing_after = df_cleaned.isna().sum().sum()
    
    cleaned_stock_data[code] = df_cleaned
    
    print(f"{code}: 缺失值 {missing_before} -> {missing_after}")

print("\n处理策略说明：")
print("采用向前填充(ffill)为主，向后填充(bfill)为辅的方法。")
print("理由：向前填充使用历史数据填补当前缺失，不会引入未来信息，")
print("对于时间序列金融数据更为合适。向后填充仅用于处理开头缺失。")

### 2.3 日期格式统一

In [ ]:
# 统一日期格式并设为索引
print("日期格式统一：")
print("-" * 60)

for code, df in cleaned_stock_data.items():
    # 检查日期列格式
    original_date_type = df['date'].dtype
    
    # 转换为datetime格式
    df['date'] = pd.to_datetime(df['date'])
    
    # 设为索引
    df.set_index('date', inplace=True)
    
    new_date_type = df.index.dtype
    print(f"{code}: 日期类型 {original_date_type} -> {new_date_type}")

print("\n所有股票数据日期已统一为datetime64格式并设为索引。")

### 2.4 数据类型检查

In [ ]:
# 检查各列数据类型
print("数据类型检查：")
print("-" * 60)

# 以第一只股票为例
first_code = list(cleaned_stock_data.keys())[0]
first_df = cleaned_stock_data[first_code]

print(f"\n示例股票 {first_code} 的数据类型：")
display(first_df.dtypes)

# 检查是否有非数值类型
print("\n数据类型转换：")
numeric_cols = ['open', 'close', 'high', 'low', 'volume', 'amount']

for code, df in cleaned_stock_data.items():
    for col in numeric_cols:
        if col in df.columns:
            if df[col].dtype == 'object':
                # 尝试转换为数值
                df[col] = pd.to_numeric(df[col], errors='coerce')
                print(f"{code} {col}: object -> float64")

print("\n所有数值列已确认为数值类型。")

### 2.5 重复值处理

In [ ]:
# 检测并删除重复行
print("重复值检测与处理：")
print("-" * 60)

total_duplicates = 0

for code, df in cleaned_stock_data.items():
    duplicates = df.index.duplicated().sum()
    if duplicates > 0:
        print(f"{code}: 发现 {duplicates} 条重复记录，已删除")
        df = df[~df.index.duplicated(keep='first')]
        cleaned_stock_data[code] = df
        total_duplicates += duplicates
    else:
        print(f"{code}: 无重复记录")

print(f"\n总共删除 {total_duplicates} 条重复记录。")

### 2.6 离群值标注

In [ ]:
# 计算日收益率并标注离群值
print("离群值标注（单日涨跌幅超过±20%）：")
print("-" * 60)

outlier_stats = []

for code, df in cleaned_stock_data.items():
    # 计算日收益率
    df['return'] = df['close'].pct_change()
    
    # 标注离群值（单日涨跌幅超过±20%）
    df['is_extreme'] = df['return'].abs() > 0.20
    
    extreme_count = df['is_extreme'].sum()
    extreme_dates = df[df['is_extreme']].index.tolist()
    
    outlier_stats.append({
        'code': code,
        'extreme_count': extreme_count,
        'extreme_dates': [str(d.date()) for d in extreme_dates[:5]]  # 最多显示5个
    })
    
    print(f"{code}: {extreme_count} 个极端波动日")

print("\n极端波动原因分析：")
print("- 涨跌停板限制：A股涨跌停限制为10%/20%，超过20%可能是新股上市")
print("- 复牌交易：长期停牌后复牌可能出现大幅波动")
print("- 除权除息：后复权数据已处理除权除息，但可能有特殊情况")
print("- 市场异常：极端市场事件（如2020年初疫情冲击）")

In [ ]:
# 显示极端波动详细信息
print("\n极端波动详细信息：")
display(pd.DataFrame(outlier_stats))

## 3. 宽表与长表转换

In [ ]:
# 创建收盘价宽表（日期为索引，每列一只股票）
print("创建收盘价宽表：")
print("-" * 60)

# 收集所有股票的收盘价
close_prices = {}

for code, df in cleaned_stock_data.items():
    close_prices[code] = df['close']

# 合并为宽表
wide_df = pd.DataFrame(close_prices)

print(f"宽表维度: {wide_df.shape}")
print("\n宽表前5行：")
display(wide_df.head())

In [ ]:
# 将宽表转换为长表
print("\n转换为长表：")
print("-" * 60)

# 使用pd.melt转换
long_df = wide_df.reset_index().melt(
    id_vars='date',
    var_name='code',
    value_name='close'
)

print(f"长表维度: {long_df.shape}")
print("\n长表前10行：")
display(long_df.head(10))

In [ ]:
# 宽表与长表的适用场景分析
print("\n宽表与长表的适用场景分析：")
print("-" * 60)
print("""
宽表（Wide Format）适用场景：
1. 多股票对比分析：直接比较不同股票在同一时点的价格
2. 相关性分析：计算股票间的相关系数矩阵
3. 可视化绑图：使用matplotlib绑制多股票走势图
4. 投资组合分析：计算组合收益率、波动率等

长表（Long Format）适用场景：
1. 数据分组操作：按股票代码分组进行统计分析
2. 数据库存储：符合数据库范式设计，便于SQL查询
3. ggplot绑图：适合使用seaborn等工具绑制分面图
4. 数据清洗：统一处理所有股票的缺失值、异常值
5. 面板数据分析：适合固定效应模型等计量分析
""")

## 4. 多表合并

In [ ]:
# 读取指数数据
print("读取指数数据：")
print("-" * 60)

index_data = {}
for idx_code in ['000300', '000905']:
    file_path = os.path.join(index_dir, f"index_{idx_code}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
        index_data[idx_code] = df
        print(f"指数 {idx_code}: {len(df)} 行")

# 使用沪深300作为市场基准
hs300 = index_data.get('000300')

In [ ]:
# 将个股数据与指数数据合并
print("\n个股与指数合并：")
print("-" * 60)

# 使用长表格式进行合并
# 首先为长表添加股票名称和行业信息
stock_info_dict = {s['code']: {'name': s['name'], 'industry': s['industry']} for s in stock_list}

long_df['name'] = long_df['code'].map(lambda x: stock_info_dict.get(x, {}).get('name', ''))
long_df['industry'] = long_df['code'].map(lambda x: stock_info_dict.get(x, {}).get('industry', ''))

# 合并指数数据
if hs300 is not None:
    # 重置索引以便合并
    long_df_merged = long_df.copy()
    
    # 添加沪深300收盘价
    hs300_reset = hs300.reset_index()
    hs300_reset = hs300_reset[['date', 'close']].rename(columns={'close': 'hs300_close'})
    
    print(f"合并前行数: {len(long_df_merged)}")
    long_df_merged = long_df_merged.merge(hs300_reset, on='date', how='left')
    print(f"合并后行数: {len(long_df_merged)}")
    
    print("\n合并后数据示例：")
    display(long_df_merged.head())

In [ ]:
# 读取宏观数据并合并
print("\n读取宏观数据：")
print("-" * 60)

macro_data = {}
for macro_name in ['cpi', 'm2']:
    file_path = os.path.join(macro_dir, f"macro_{macro_name}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"{macro_name.upper()}: {len(df)} 行")
        print(f"  列: {df.columns.tolist()}")
        print(f"  示例: {df.head(2).to_dict('records')}")
        macro_data[macro_name] = df

In [ ]:
# 处理宏观数据的月度频率与日度数据的合并
print("\n宏观数据与日度数据合并：")
print("-" * 60)

# 将日期转换为月份格式
long_df_merged['year_month'] = long_df_merged['date'].dt.strftime('%Y-%m')

# 合并CPI数据
if 'cpi' in macro_data:
    cpi_df = macro_data['cpi'].copy()
    # 检查日期列名
    date_col = 'date' if 'date' in cpi_df.columns else cpi_df.columns[0]
    value_col = 'cpi_yoy' if 'cpi_yoy' in cpi_df.columns else cpi_df.columns[1]
    
    cpi_df = cpi_df.rename(columns={date_col: 'year_month', value_col: 'cpi_yoy'})
    
    # 确保year_month格式一致
    if 'year_month' in cpi_df.columns:
        # 可能需要处理日期格式
        cpi_df['year_month'] = pd.to_datetime(cpi_df['year_month']).dt.strftime('%Y-%m')
        
        print(f"合并前: {len(long_df_merged)} 行")
        long_df_merged = long_df_merged.merge(
            cpi_df[['year_month', 'cpi_yoy']], 
            on='year_month', 
            how='left'
        )
        print(f"合并CPI后: {len(long_df_merged)} 行")

# 合并M2数据
if 'm2' in macro_data:
    m2_df = macro_data['m2'].copy()
    date_col = 'date' if 'date' in m2_df.columns else m2_df.columns[0]
    value_col = 'm2_yoy' if 'm2_yoy' in m2_df.columns else m2_df.columns[1]
    
    m2_df = m2_df.rename(columns={date_col: 'year_month', value_col: 'm2_yoy'})
    
    if 'year_month' in m2_df.columns:
        m2_df['year_month'] = pd.to_datetime(m2_df['year_month']).dt.strftime('%Y-%m')
        
        long_df_merged = long_df_merged.merge(
            m2_df[['year_month', 'm2_yoy']], 
            on='year_month', 
            how='left'
        )
        print(f"合并M2后: {len(long_df_merged)} 行")

print("\n合并后数据列：")
print(long_df_merged.columns.tolist())

print("\n合并后数据示例：")
display(long_df_merged.head())

In [ ]:
# 计算日收益率
print("\n计算日收益率：")
print("-" * 60)

# 按股票分组计算收益率
long_df_merged = long_df_merged.sort_values(['code', 'date'])
long_df_merged['return'] = long_df_merged.groupby('code')['close'].pct_change()

# 计算沪深300收益率
long_df_merged['hs300_return'] = long_df_merged.groupby('code')['hs300_close'].pct_change()

print("收益率计算完成！")
print("\n最终合并数据示例：")
display(long_df_merged.head(10))

## 5. 数据存储

### 5.1 基础要求：CSV格式

In [ ]:
# 保存清洗后的数据为CSV
print("保存清洗后的数据（CSV格式）：")
print("-" * 60)

# 保存股票清洗数据
stock_clean_list = []
for code, df in cleaned_stock_data.items():
    df_save = df.reset_index()
    df_save['code'] = code
    df_save['name'] = stock_info_dict.get(code, {}).get('name', '')
    df_save['industry'] = stock_info_dict.get(code, {}).get('industry', '')
    stock_clean_list.append(df_save)

stock_clean_df = pd.concat(stock_clean_list, ignore_index=True)
stock_clean_path = os.path.join(clean_dir, "stock_clean.csv")
stock_clean_df.to_csv(stock_clean_path, index=False, encoding='utf-8')
print(f"股票清洗数据已保存: {stock_clean_path}")
print(f"  文件大小: {os.path.getsize(stock_clean_path) / 1024:.1f} KB")

# 保存合并数据
combined_path = os.path.join(combined_dir, "combined_data.csv")
long_df_merged.to_csv(combined_path, index=False, encoding='utf-8')
print(f"\n合并数据已保存: {combined_path}")
print(f"  文件大小: {os.path.getsize(combined_path) / 1024:.1f} KB")

### 5.2 进阶要求：Parquet格式

In [ ]:
# 保存为Parquet格式
print("保存清洗后的数据（Parquet格式）：")
print("-" * 60)

parquet_path = os.path.join(clean_dir, "stock_clean.parquet")
stock_clean_df.to_parquet(parquet_path, index=False)
print(f"Parquet文件已保存: {parquet_path}")
print(f"  文件大小: {os.path.getsize(parquet_path) / 1024:.1f} KB")

In [ ]:
# 演示Parquet特性
print("\nParquet特性演示：")
print("-" * 60)

# 1. 列式读取（只加载需要的列）
print("\n1. 列式读取（只加载需要的列）：")
df_partial = pd.read_parquet(parquet_path, columns=["date", "code", "close"])
print(f"读取的列: {df_partial.columns.tolist()}")
print(f"前5行:")
display(df_partial.head())

In [ ]:
# 2. 查看Schema（类型契约）
print("\n2. 查看Schema（类型契约）：")
schema = pq.read_schema(parquet_path)
print(schema)

In [ ]:
# 3. 与CSV对比：读取速度与文件体积
print("\n3. CSV vs Parquet 对比：")
print("-" * 60)

# CSV读取
t0 = time.time()
df_csv = pd.read_csv(stock_clean_path)
csv_time = time.time() - t0
csv_size = os.path.getsize(stock_clean_path) / 1024

# Parquet读取
t0 = time.time()
df_parquet = pd.read_parquet(parquet_path)
parquet_time = time.time() - t0
parquet_size = os.path.getsize(parquet_path) / 1024

print(f"CSV:    读取耗时 {csv_time:.4f}s, 文件大小 {csv_size:.1f} KB")
print(f"Parquet: 读取耗时 {parquet_time:.4f}s, 文件大小 {parquet_size:.1f} KB")
print(f"\n体积压缩率: {(1 - parquet_size/csv_size)*100:.1f}%")

In [ ]:
# Parquet与CSV对比分析
print("\nParquet与CSV对比分析：")
print("-" * 60)
print(f"""
在本次数据规模下（约{len(stock_clean_df)}行数据）：

1. 文件体积：Parquet比CSV小约{(1 - parquet_size/csv_size)*100:.1f}%
   - Parquet采用列式压缩，对于重复值较多的列压缩效果更好
   - 在大数据场景下，体积差异会更加显著

2. 读取速度：
   - 在小数据集上差异不明显
   - 当数据量达到GB级别时，Parquet的列式读取优势明显
   - 如果只需要读取部分列，Parquet可以只加载需要的列，速度更快

3. 类型安全：
   - Parquet保存了Schema信息，读取时自动恢复正确类型
   - CSV需要手动处理类型转换

4. 适用场景：
   - CSV：适合小数据集、人工查看、跨平台兼容
   - Parquet：适合大数据分析、只读部分列、需要类型安全

选择Parquet的理由：
- 列式存储更适合分析型查询
- 更好的压缩率节省存储空间
- 保留了数据类型信息，减少数据清洗工作
""")

## 6. 数据清洗总结

In [ ]:
# 数据清洗总结
print("数据清洗总结：")
print("=" * 60)

summary = {
    '清洗步骤': [
        '缺失值检测',
        '缺失值处理',
        '日期格式统一',
        '数据类型检查',
        '重复值处理',
        '离群值标注'
    ],
    '处理方法': [
        '统计每列缺失数量和比例',
        '向前填充(ffill)为主，向后填充(bfill)为辅',
        '转换为datetime64格式并设为索引',
        '确认所有数值列为float64类型',
        '检测并删除重复行',
        '计算日收益率，标注|收益|>20%为极端值'
    ],
    '结果': [
        f'总缺失值: {total_missing}',
        '缺失值已填充',
        '日期格式统一',
        '数据类型正确',
        f'删除{total_duplicates}条重复记录',
        f'共发现{sum([s["extreme_count"] for s in outlier_stats])}个极端波动日'
    ]
}

summary_df = pd.DataFrame(summary)
display(summary_df)

print("\n数据清洗完成！")
print("下一步：运行 03_analysis.ipynb 进行描述统计与回归分析")